In [1]:
import psycopg2
from psycopg2 import Error


In [2]:

# --- Paramètres de connexion ---
user = 'mngerscrpdb'
password = 'P@ssw0rd'
host = '192.168.1.248'
port = '5432'
database = 'scrappinjob_db'
# -----------------------------


In [12]:
def execute_quality_audit(conn):
    """
    Exécute un audit complet de qualité des données sur la base de données PostgreSQL 
    en comptant les NULLs, les chaînes vides, les valeurs distinctes et la cardinalité 
    pour toutes les tables du schéma 'public'. Gère la vérification des chaînes vides 
    uniquement pour les types de données textuelles.
    """
    cursor = conn.cursor()
    audit_results = {}

    try:
        # 1. Récupérer les métadonnées (table, colonne et type de données)
        print("Récupération de la structure et des types des tables...")
        cursor.execute("""
            SELECT 
                table_name, 
                column_name,
                data_type
            FROM 
                information_schema.columns
            WHERE 
                table_schema = 'public'
            ORDER BY 
                table_name, ordinal_position;
        """)
        
        tables_and_columns_with_types = cursor.fetchall()
        
        # Regrouper les colonnes par table, incluant le type
        schema = {}
        for table, column, data_type in tables_and_columns_with_types:
            if table not in schema:
                schema[table] = []
            # Stocker un tuple (nom_colonne, type_colonne)
            schema[table].append((column, data_type)) 

        print(f"Audit en cours sur {len(schema)} tables...")

        # 2. Itérer sur chaque table pour générer et exécuter la requête d'audit
        for table_name, columns_and_types in schema.items():
            audit_results[table_name] = []
            select_parts = []
            
            for col, data_type in columns_and_types:
                
                # Nettoyer le nom de la colonne pour l'utiliser comme ALIAS SQL (gère les espaces et %)
                safe_alias = re.sub(r'[^a-zA-Z0-9_]', '_', col) 
                
                # Compteur 1 : Valeurs NULL (Fonctionne pour TOUS les types)
                select_parts.append(
                    f"COUNT(CASE WHEN \"{col}\" IS NULL THEN 1 END) AS null_count_{safe_alias}"
                )
                
                # Compteur 2 : Chaînes Vides ('') - Conditionnel au type de données
                # Les types textuels courants sont : text, character varying, char.
                if data_type in ('text', 'character varying', 'char', 'varchar'):
                    # Utiliser TRIM() pour capturer les chaînes contenant uniquement des espaces
                    empty_count_clause = f"COUNT(CASE WHEN \"{col}\" = '' OR TRIM(\"{col}\") = '' THEN 1 END)"
                else:
                    # Si c'est un type numérique, date, booléen, etc., le résultat du comptage est 0
                    empty_count_clause = "0" 

                select_parts.append(f"{empty_count_clause} AS empty_string_count_{safe_alias}")
                
                # Compteur 3 : Valeurs Distinctes (Cardinalité)
                select_parts.append(
                    f"COUNT(DISTINCT \"{col}\") AS distinct_count_{safe_alias}"
                )

            # Ajouter le nombre total de lignes
            select_parts.append("COUNT(*) AS total_rows")
            
            # Assembler la requête finale
            query = f"""
                SELECT 
                    {', '.join(select_parts)}
                FROM 
                    public.\"{table_name}\";
            """

            # Exécuter la requête
            cursor.execute(query)
            result = cursor.fetchone()
            
            # 3. Stocker les résultats
            total_rows = result[-1]
            
            # Le résultat a 3 entrées par colonne (NULL, EMPTY, DISTINCT) + 1 (total_rows)
            for i, (col, data_type) in enumerate(columns_and_types):
                
                null_count = result[i * 3]
                empty_string_count = result[i * 3 + 1]
                distinct_count = result[i * 3 + 2]
                
                # Calcul des pourcentages (Évite la division par zéro)
                null_percentage = (null_count / total_rows * 100) if total_rows > 0 else 0
                empty_percentage = (empty_string_count / total_rows * 100) if total_rows > 0 else 0
                cardinality_rate = (distinct_count / total_rows * 100) if total_rows > 0 else 0

                audit_results[table_name].append({
                    'column': col,
                    'data_type': data_type,
                    'null_count': null_count,
                    'null_percentage': f"{null_percentage:.2f}%",
                    'empty_count': empty_string_count,
                    'empty_percentage': f"{empty_percentage:.2f}%",
                    'distinct_count': distinct_count,
                    'cardinality_rate': f"{cardinality_rate:.2f}%",
                    'total_rows': total_rows
                })

        return audit_results

    except (Exception, Error) as error:
        print(f"❌ Une erreur s'est produite pendant l'audit : {error}")
        return None
    finally:
        if 'cursor' in locals() and cursor:
            cursor.close()

# ===============================================
# 3. BLOC PRINCIPAL D'EXÉCUTION ET D'AFFICHAGE
# ===============================================

if __name__ == "__main__":
    conn = None
    try:
        # Établissement de la connexion
        conn = psycopg2.connect(
            user=user,
            password=password,
            host=host,
            port=port,
            database=database
        )
        print("✅ Connexion établie.")

        # Lancement de l'audit
        audit_data = execute_quality_audit(conn)

        if audit_data:
            print("\n" + "="*80)
            print("--- 📊 RÉSULTATS DE L'AUDIT DE QUALITÉ DES DONNÉES ---")
            print("="*80)
            
            # Titres de colonnes pour l'affichage
            table_header = "| Colonne | Type | Lignes | NULL (%) | Vide (%) | Distincts | Cardinalité (%) |"
            table_separator = "| :--- | :---: | :---: | :---: | :---: | :---: | :---: |"
            
            for table, results in audit_data.items():
                if not results:
                    continue
                    
                total_rows = results[0]['total_rows']
                print(f"\n## Table : **{table}**")

                # Affichage des résultats sous forme de tableau Markdown
                print(table_header)
                print(table_separator)
                
                for item in results:
                    print(
                        f"| {item['column']} | {item['data_type']} | {item['total_rows']} "
                        f"| {item['null_percentage']} | {item['empty_percentage']} "
                        f"| {item['distinct_count']} | **{item['cardinality_rate']}** |"
                    )
                    
            print("\n" + "="*80)
            print("--- Fin de l'Audit ---")
            print("="*80)

    except (Exception, Error) as error:
        print(f"❌ Erreur critique de connexion : {error}")

    finally:
        if conn:
            conn.close()
            print("\n🔒 Connexion PostgreSQL fermée.")

✅ Connexion établie.
Récupération de la structure et des types des tables...
Audit en cours sur 34 tables...

--- 📊 RÉSULTATS DE L'AUDIT DE QUALITÉ DES DONNÉES ---

## Table : **auth_group**
| Colonne | Type | Lignes | NULL (%) | Vide (%) | Distincts | Cardinalité (%) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| id | integer | 0 | 0.00% | 0.00% | 0 | **0.00%** |
| name | character varying | 0 | 0.00% | 0.00% | 0 | **0.00%** |

## Table : **auth_group_permissions**
| Colonne | Type | Lignes | NULL (%) | Vide (%) | Distincts | Cardinalité (%) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| id | bigint | 0 | 0.00% | 0.00% | 0 | **0.00%** |
| group_id | integer | 0 | 0.00% | 0.00% | 0 | **0.00%** |
| permission_id | integer | 0 | 0.00% | 0.00% | 0 | **0.00%** |

## Table : **auth_permission**
| Colonne | Type | Lignes | NULL (%) | Vide (%) | Distincts | Cardinalité (%) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| id | integer | 44 | 0.00% | 0.00